Cell 1 imports all needed libraries and functions. Cell 2 imports tooth morphometric data from a spreadsheet. Cell 3 size-standardizes and log transforms the data. Cell 4 makes the correlation matrix. Cell 5 performs the PCA and displays principal components, their eigenvalues, loadings, and variance explained. Cell 6 displays the PCA plot. Cell 7 performs the PERMANOVA, both global and pairwise. Comments are placed throughout all the cells. The author would like to acknowledge that Claude AI assissted in debugging the code. 


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial import ConvexHull

# For displaying dataframes 
from IPython.display import display

# Plot styling
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("All libraries imported successfully!")

In [ ]:
df = pd.read_csv('PUT FILE NAME HERE')

# Column mapping
column_mapping = {
    'FABL/CBL (mm)': 'CBL',
    'CBW (mm)': 'CBW',
    'AL (mm)': 'AL',
    'CH (mm)': 'CH',
    'MCL (mm)': 'MCL',
    'MCW (mm)': 'MCW',
    'MDD(per 5mm)': 'MDD',
    'DDD (per 5mm)': 'DDD',
    'CBR': 'CBR',
    'Taxa': 'Taxa'
}

# Applies mapping
df.rename(columns=column_mapping, inplace=True)                   

print(f"Total samples: {len(df)}") # Print total number of samples
print(f"\nColumns in dataset:")   
print(list(df.columns))
print(f"\nSamples per taxa:")   # List number of samples per taxa
print(df['Taxa'].value_counts())      

In [ ]:
print("DATA TRANSFORMATION")

# Create a copy for transformation
df_transformed = df.copy()

# Size-related variables (linear measurements) - divide by CBL
size_variables = ['CBW', 'AL', 'CH', 'MCL', 'MCW']

# Non-size variables (densities and ratios) - do not divide by CBL
non_size_variables = ['MDD', 'DDD', 'CBR']

# Only include variables that exist in the dataset
available_size_vars = [var for var in size_variables if var in df.columns]         
available_nonsize_vars = [var for var in non_size_variables if var in df.columns] 

# Size standardize size-related variables (divide by CBL) 
for var in available_size_vars:
    df_transformed[f'{var}_ratio'] = df[var] / df['CBL']

# Keep non-size variables the same
for var in available_nonsize_vars:
    df_transformed[f'{var}_ratio'] = df[var]

# Log-transform all variables
all_available_vars = available_size_vars + available_nonsize_vars
for var in all_available_vars:
    ratio_col = f'{var}_ratio'
    # Check for positive values before log transform
    if df_transformed[ratio_col].min() > 0:
        df_transformed[f'log_{var}'] = np.log(df_transformed[ratio_col])
    else:
        print(f"Warning: {var} contains non-positive values, skipping log transform")

log_cols = [f'log_{var}' for var in all_available_vars if f'log_{var}' in df_transformed.columns]

# Extract transformed data for PCA
transformed_vars = log_cols
X_transformed = df_transformed[transformed_vars].values

# Print out number of variables for PCA  
print(f"Variables for Correlation PCA: {len(transformed_vars)} variables")

In [ ]:
# Make correlation matrix of transformed variables
corr_matrix = df_transformed[transformed_vars].corr()

print("\nCorrelation Matrix (log-transformed variables):")
display(corr_matrix.round(3))  # Round to 3 digits

# Display correlation matrix
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix of Transformed Variables', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
print("CORRELATION PCA")

# Standardize the data (z-score normalization) for correlation PCA
scaler = StandardScaler()
X_standardized = scaler.fit_transform(X_transformed)

# Perform PCA
pca_corr = PCA()
pc_scores_corr = pca_corr.fit_transform(X_standardized)

# Eigenvalues
eigenvalues_corr = pca_corr.explained_variance_
print("\nEigenvalues:")
for i, ev in enumerate(eigenvalues_corr[:7]):  # Show the eigenvalues of the 7 PCs
    print(f"  PC{i+1}: {ev:.4f}")

# Explained variance
explained_var_corr = pca_corr.explained_variance_ratio_
print("\nExplained Variance Ratio:")
for i, var in enumerate(explained_var_corr[:7]):  # Show the 7 PCs explaining total variance
    print(f"  PC{i+1}: {var*100:.2f}%")
cumulative = sum(explained_var_corr[:2])
print(f"\n  Cumulative (PC1-PC2): {cumulative*100:.2f}%")

# Loadings (correlation between original variables and PCs)
loadings_corr = pca_corr.components_.T
loadings_df_corr = pd.DataFrame(
    loadings_corr,
    columns=[f'PC{i+1}' for i in range(len(transformed_vars))],
    index=transformed_vars
)

print("\nPCA Loadings (first 7 PCs):")
display(loadings_df_corr.iloc[:, :7].round(4))

# Add PC scores to dataframe
df_transformed['PC1_corr'] = pc_scores_corr[:, 0]
df_transformed['PC2_corr'] = pc_scores_corr[:, 1]
if pc_scores_corr.shape[1] > 2:
    df_transformed['PC3_corr'] = pc_scores_corr[:, 2]

In [ ]:
print("CORRELATION PCA GRAPH")

# Define colours for each Taxa
Taxa_colours = {
    'Ankylosaurid': "#d703f8",
    'Nodosaurid': "#ff7b00",
    'Pachycephalosaurid': '#4daf4a',
    'Troodontid': "#008cff",
    'Dromaeosaurid': "#ff0004",
    'Tyrannosaurid': "#F9E002",
    'Hyphsilophidontid': "#213A21"
}

# Get actual families in your data and assign colors
actual_taxa = df_transformed['Taxa'].unique()
taxa_colour_map = {}
for Taxa in actual_taxa:
    if Taxa in Taxa_colours:
        taxa_colour_map[Taxa] = Taxa_colours[Taxa]
    else:
        # Assign default color if not in predefined list
        taxa_colour_map[Taxa] = f"C{len(taxa_colour_map)}"

fig, ax = plt.subplots(figsize=(12, 10))

# Plot each family with different colors
for taxa in taxa_colour_map:
    mask = df_transformed['Taxa'] == taxa
    ax.scatter(df_transformed.loc[mask, 'PC1_corr'],
               df_transformed.loc[mask, 'PC2_corr'],
               c=taxa_colour_map[taxa], label=taxa, s=100, alpha=0.7, 
               edgecolors='black', linewidth=1.5)
    
    #  Add convex hull for each family                                               
    points = df_transformed.loc[mask, ['PC1_corr', 'PC2_corr']].values
    if len(points) >= 3:
        try:
            hull = ConvexHull(points)
            for simplex in hull.simplices:
                ax.plot(points[simplex, 0], points[simplex, 1],
                       c=taxa_colour_map[taxa], alpha=0.3, linewidth=2)
            # Fill the hull
            hull_points = points[hull.vertices]
            ax.fill(hull_points[:, 0], hull_points[:, 1],
                   c=taxa_colour_map[taxa], alpha=0.1)
        except:
            # Skip hull if points are collinear
            pass

# Plot eigenvectors (loadings) as arrows
arrow_scale = 6.0  # Adjust this if arrows are too long/short

for i, var in enumerate(transformed_vars):
    ax.annotate('', xy=(loadings_corr[i, 0] * arrow_scale, loadings_corr[i, 1] * arrow_scale),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Simplify variable names for plot (remove 'log_' prefix)
    var_name = var.replace('log_', '')
    ax.text(loadings_corr[i, 0] * arrow_scale * 1.2,
            loadings_corr[i, 1] * arrow_scale * 1.2,
            var_name, fontsize=10, fontweight='bold', ha='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# Labels and formatting
ax.set_xlabel(f'PC1 ({explained_var_corr[0]*100:.1f}% variance)', fontsize=13, fontweight='bold')
ax.set_ylabel(f'PC2 ({explained_var_corr[1]*100:.1f}% variance)', fontsize=13, fontweight='bold')
ax.set_title('Correlation PCA graph',
             fontsize=14, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.5)
ax.legend(loc='best', fontsize=9)
ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.show()

In [ ]:
print("PERMANOVA")

def permanova_test(X, grouping, permutations=9999):
    """
    PERMANOVA using Euclidean distances.
    Returns pseudo-F statistic and p-value.
    """
    dist_matrix = squareform(pdist(X, metric='euclidean'))
    groups = np.array(grouping)
    n = len(groups)
    
    def pseudo_f(dist_sq, grps):
        unique_groups = np.unique(grps)
        N = len(grps)
        a = len(unique_groups)
        
        # Total sum of squares
        SS_T = np.sum(dist_sq) / N
        
        # Within-group sum of squares
        SS_W = 0
        for g in unique_groups:
            idx = np.where(grps == g)[0]
            n_g = len(idx)
            if n_g > 1:
                sub = dist_sq[np.ix_(idx, idx)]
                SS_W += np.sum(sub) / n_g
        
        SS_A = SS_T - SS_W
        df_A = a - 1
        df_W = N - a
        
        if SS_W == 0 or df_W == 0:
            return np.nan
        return (SS_A / df_A) / (SS_W / df_W)
    
    dist_sq = dist_matrix ** 2
    observed_f = pseudo_f(dist_sq, groups)
    
    # Permutation test
    count = 0
    for _ in range(permutations):
        perm_groups = np.random.permutation(groups)
        perm_f = pseudo_f(dist_sq, perm_groups)
        if perm_f >= observed_f:
            count += 1
    
    p_value = (count + 1) / (permutations + 1)
    return observed_f, p_value


# --- Global PERMANOVA ---
grouping = df_transformed['Taxa'].reset_index(drop=True).values
obs_f, p_val = permanova_test(X_standardized, grouping, permutations=9999)

print("\n=== PERMANOVA Results ===")
print(f"Test statistic (pseudo-F): {obs_f:.4f}")
print(f"p-value (9999 permutations): {p_val:.4f}")

if p_val < 0.05:
    print("\n→ Significant difference in multivariate morphospace among taxa (p < 0.05)")
else:
    print("\n→ No significant difference detected among taxa (p ≥ 0.05)")


# --- Pairwise PERMANOVA ---
print("\n=== Pairwise PERMANOVA (all taxon pairs) ===")

taxa_list = df_transformed['Taxa'].unique()
pairwise_results = []

for taxa1, taxa2 in combinations(taxa_list, 2):
    mask = df_transformed['Taxa'].isin([taxa1, taxa2]).values
    sub_X = X_standardized[mask]
    sub_grouping = df_transformed.loc[mask, 'Taxa'].values
    
    if len(np.unique(sub_grouping)) < 2 or len(sub_grouping) < 4:
        continue  # Skip if too few samples
    
    f_stat, p = permanova_test(sub_X, sub_grouping, permutations=9999)
    pairwise_results.append({
        'Taxa 1': taxa1,
        'Taxa 2': taxa2,
        'pseudo-F': round(f_stat, 4),
        'p-value': round(p, 4),
        'significant': '*' if p < 0.05 else 'ns'
    })

pairwise_df = pd.DataFrame(pairwise_results)
display(pairwise_df)